# Generate the Customer Accounts Dataset

This notebook creates `data/customer_accounts.csv`, the synthetic dataset used by every
lesson in this stage. The file is already included, so you only need to run this notebook
if you want to recreate it (run all cells, top to bottom).

Every account is **made up** with a fixed random seed. There is no real customer data.

## What is in the file

One row per customer account (2,000 accounts).

| Column | Meaning |
|--------|---------|
| `account_id` | Opaque id such as `A0001` |
| `segment` | Hospital, Clinic, Distributor, or Laboratory |
| `region` | North America, Europe, Asia Pacific, or Latin America |
| `years_as_customer` | Whole years since the account opened |
| `on_contract` | 1 if the account has a multi-year contract, else 0 |
| `orders_last_year` | Number of orders placed last year |
| `avg_order_value` | Average order value, USD |
| `discount_pct` | Average discount given, percent |
| `support_tickets` | Support tickets raised last year |
| `avg_days_to_pay` | Average days taken to pay an invoice last year |
| `spend_last_year` | Total spend last year, USD |
| `next_year_spend` | Spend in the following year, USD (a **regression** target) |
| `paid_late` | 1 if the account paid late this year, else 0 (a **classification** target) |

A few blemishes are included on purpose for the preprocessing lesson: some missing
`avg_days_to_pay` and `discount_pct` values, and some regions written in lower case.

In [ ]:
import os
import numpy as np
import pandas as pd

os.makedirs("data", exist_ok=True)
rng = np.random.default_rng(7)
n = 2000

SEGMENTS = ["Hospital", "Clinic", "Distributor", "Laboratory"]
REGIONS = ["North America", "Europe", "Asia Pacific", "Latin America"]
segment = rng.choice(SEGMENTS, n, p=[0.30, 0.35, 0.20, 0.15])
region = rng.choice(REGIONS, n, p=[0.40, 0.30, 0.20, 0.10])

# Each segment buys differently: typical orders per year and order value (USD).
ORDERS = {"Hospital": 40, "Clinic": 18, "Distributor": 60, "Laboratory": 25}
ORDER_VALUE = {"Hospital": 8000, "Clinic": 2500, "Distributor": 12000, "Laboratory": 4000}

In [ ]:
years = rng.integers(0, 16, n)
on_contract = (rng.random(n) < np.clip(0.30 + 0.03 * years, 0, 0.85)).astype(int)
orders = rng.poisson([ORDERS[s] for s in segment])
order_value = rng.lognormal(np.log([ORDER_VALUE[s] for s in segment]), 0.35)
is_distributor = (segment == "Distributor").astype(int)
discount = np.clip(rng.normal(5 + 5 * is_distributor + 2 * on_contract, 2), 0, 25)
tickets = rng.poisson(1 + orders / 20)
days_to_pay = np.clip(rng.normal(35 + 10 * is_distributor + 8 * (region == "Latin America")
                                 - 0.8 * years - 5 * on_contract, 8), 10, 120)
spend = orders * order_value

# Next year's spend grows with contracts and loyalty, and shrinks with support problems.
growth = (0.04 + 0.05 * on_contract + 0.005 * years - 0.01 * tickets
          + rng.normal(0, 0.08, n))
next_spend = spend * (1 + growth)

# The chance of paying late rises with slow past payment and tickets, and falls with
# contracts and loyalty.
z = (-1.2 + 0.06 * (days_to_pay - 35) + 0.15 * tickets - 0.9 * on_contract
     - 0.05 * years + 0.6 * is_distributor)
paid_late = (rng.random(n) < 1 / (1 + np.exp(-z))).astype(int)

In [ ]:
df = pd.DataFrame({
    "account_id": [f"A{i:04d}" for i in range(1, n + 1)],
    "segment": segment,
    "region": region,
    "years_as_customer": years,
    "on_contract": on_contract,
    "orders_last_year": orders,
    "avg_order_value": order_value.round(0),
    "discount_pct": discount.round(1),
    "support_tickets": tickets,
    "avg_days_to_pay": days_to_pay.round(1),
    "spend_last_year": spend.round(0),
    "next_year_spend": next_spend.round(0),
    "paid_late": paid_late,
})

# Blemishes for the preprocessing lesson.
df.loc[df.sample(60, random_state=1).index, "avg_days_to_pay"] = np.nan
df.loc[df.sample(40, random_state=2).index, "discount_pct"] = np.nan
lower = df.sample(25, random_state=3).index
df.loc[lower, "region"] = df.loc[lower, "region"].str.lower()

path = os.path.join("data", "customer_accounts.csv")
df.to_csv(path, index=False)
print(f"wrote {path}: {len(df)} rows, late payment rate {df['paid_late'].mean():.1%}")